# 02 — Pré-processamento: Normalização Temporal T=100

**Objetivo:** Carregar o manifest.csv (gerado no notebook 01), filtrar amostras válidas,
normalizar todas as sequências para T=100 frames via interpolação linear, e salvar
o dataset final em formato `.npz`.

**Estratégia de normalização:**
- Se T_orig > 100: subamostragem por interpolação linear
- Se T_orig < 100: upsampling por interpolação linear  
- Se T_orig == 100: mantém como está
- NaNs são preenchidos por interpolação linear antes da normalização

**Output:**
- `data/ravdess_landmarks_kaggle/01_processed_T100/dataset_T100.npz`
  - X: shape (N, 100, D)
  - y: shape (N,) — emotion code 0-indexed
  - actor_ids: shape (N,)

In [ ]:
import sys, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.ravdess_utils import load_landmark_csv, EMOTION_MAP, EMOTION_LABELS
from src.temporal import normalize_sequence_length, build_dataset_T100

print("Imports OK")

## 2.1 Carregar Manifest

In [ ]:
QC_DIR = os.path.join(ROOT, 'data', 'ravdess_landmarks_kaggle', '03_qc')
OUT_DIR = os.path.join(ROOT, 'data', 'ravdess_landmarks_kaggle', '01_processed_T100')
os.makedirs(OUT_DIR, exist_ok=True)

manifest = pd.read_csv(os.path.join(QC_DIR, 'manifest.csv'))
print(f"Manifest carregado: {manifest.shape}")

# Filtrar amostras válidas
valid = manifest[manifest['status_ok'] == True].copy()
print(f"Amostras válidas: {len(valid)}")
valid.head()

## 2.2 Sanity Check: inspecionar uma amostra antes da normalização

In [ ]:
# Carregar primeira amostra para inspeção
sample_row = valid.iloc[0]
sample_df = load_landmark_csv(sample_row['filepath'])
numeric_cols = sample_df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Arquivo: {sample_row['filename']}")
print(f"Emoção: {sample_row['emotion_label']} (code={sample_row['emotion_code']})")
print(f"Ator: {sample_row['actor_id']}")
print(f"Frames: {len(sample_df)}")
print(f"Features numéricas: {len(numeric_cols)}")
print(f"\nPrimeiras 5 colunas numéricas: {numeric_cols[:5]}")

sample_df[numeric_cols].describe()

## 2.3 Construir Dataset T=100

In [ ]:
%%time
TARGET_T = 100

X, y, meta = build_dataset_T100(
    manifest_df=valid,
    load_csv_fn=load_landmark_csv,
    target_len=TARGET_T,
    verbose=True,
)

print(f"\nDataset final:")
print(f"  X shape: {X.shape}  (N={X.shape[0]}, T={X.shape[1]}, D={X.shape[2]})")
print(f"  y shape: {y.shape}")
print(f"  y classes únicas: {np.unique(y)}")
print(f"  dtype X: {X.dtype}, dtype y: {y.dtype}")

In [ ]:
# Extrair actor_ids do metadata
actor_ids = np.array([m['actor_id'] for m in meta], dtype=np.int64)
emotion_labels_list = [m['emotion_label'] for m in meta]

# Verificar distribuição de classes
print("Distribuição de classes no dataset final:")
for code in sorted(np.unique(y)):
    label = EMOTION_MAP.get(code + 1, f"unknown_{code}")
    count = (y == code).sum()
    print(f"  {code} ({label}): {count} amostras")

## 2.4 Sanity Check: valores do dataset

In [ ]:
# Checks básicos
print("Sanity checks:")
print(f"  NaN no X? {np.isnan(X).any()}")
print(f"  Inf no X? {np.isinf(X).any()}")
print(f"  X min: {X.min():.4f}")
print(f"  X max: {X.max():.4f}")
print(f"  X mean: {X.mean():.4f}")
print(f"  X std: {X.std():.4f}")

# Verificar se todas as amostras têm o mesmo número de features
print(f"\n  Todas as amostras com T={TARGET_T}? {X.shape[1] == TARGET_T}")
print(f"  D (features): {X.shape[2]}")

## 2.5 Visualizações: trajetórias temporais de landmarks

In [ ]:
# Visualizar trajetória temporal de algumas features para diferentes emoções
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

# Escolher 1 amostra de cada emoção
n_features_to_plot = 3  # Mostrar 3 features por amostra

for emo_idx in range(min(8, len(np.unique(y)))):
    # Encontrar primeira amostra desta emoção
    mask = y == emo_idx
    if not mask.any():
        continue
    sample_idx = np.where(mask)[0][0]
    sample = X[sample_idx]  # (T, D)
    label = EMOTION_MAP.get(emo_idx + 1, f"emo_{emo_idx}")

    ax = axes[emo_idx]
    for feat_i in range(min(n_features_to_plot, sample.shape[1])):
        ax.plot(sample[:, feat_i], alpha=0.7, label=f"feat_{feat_i}")
    ax.set_title(f"{label} (idx={sample_idx})")
    ax.set_xlabel('Frame (T=100)')
    ax.set_ylabel('Valor')
    if emo_idx == 0:
        ax.legend(fontsize=7)

plt.suptitle('Trajetórias Temporais por Emoção (3 features)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar efeito da normalização temporal em uma amostra
sample_row = valid.iloc[0]
df_orig = load_landmark_csv(sample_row['filepath'])
numeric_cols = df_orig.select_dtypes(include=[np.number]).columns.tolist()
orig_vals = df_orig[numeric_cols].values

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

feat_idx = 0

axes[0].plot(orig_vals[:, feat_idx], 'b-o', markersize=2)
axes[0].set_title(f'Original (T={orig_vals.shape[0]})')
axes[0].set_xlabel('Frame')
axes[0].set_ylabel(f'Feature {numeric_cols[feat_idx]}')

# Versão normalizada
norm_vals = normalize_sequence_length(orig_vals, TARGET_T)
axes[1].plot(norm_vals[:, feat_idx], 'r-o', markersize=2)
axes[1].set_title(f'Normalizado (T={TARGET_T})')
axes[1].set_xlabel('Frame')
axes[1].set_ylabel(f'Feature {numeric_cols[feat_idx]}')

plt.suptitle(f'Efeito da Normalização Temporal — {sample_row["emotion_label"]}', fontsize=12)
plt.tight_layout()
plt.show()

## 2.6 Salvar Dataset

In [ ]:
# Salvar em formato .npz
out_path = os.path.join(OUT_DIR, 'dataset_T100.npz')
np.savez_compressed(
    out_path,
    X=X,
    y=y,
    actor_ids=actor_ids,
    emotion_labels=np.array(emotion_labels_list),
)

# Verificar tamanho
size_mb = os.path.getsize(out_path) / (1024 * 1024)
print(f"Dataset salvo em: {out_path}")
print(f"Tamanho: {size_mb:.1f} MB")
print(f"Shape: X={X.shape}, y={y.shape}, actor_ids={actor_ids.shape}")

print("\nNotebook 02 concluído com sucesso!")